# Intrinsic Value - AI Allocation Portfolio

AlphaSpread-style intrinsic value vs market price, one tab per stock.

**Method (calibrated to AlphaSpread's Base Case).** Intrinsic value blends
two legs, **80% multiples + 20% DCF**:

- **Multiples leg (dominant):** forward EPS x a *risk-adjusted* fair P/E.
  The fair P/E starts at ~24x (beta = 1) and shrinks as beta rises, so
  higher-risk names are valued more cautiously. This is what lets the model
  call low-beta names undervalued and high-beta names overvalued at similar
  forward P/Es.
- **DCF leg:** a 2-stage discounted cash flow on *normalized* (multi-year
  average) free cash flow, so cyclical peak years don't inflate value.

Parameters were grid-search fit against five AlphaSpread anchors (CCJ, MSFT,
NVDA, BESI.AS, ALAB); the fit is direction-correct on all five with ~20%
RMSE. Names with neither positive FCF nor positive EPS are marked **N/A**.

Run the cell below.

In [ ]:
import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from IPython.display import HTML, display

from portfolio.intrinsic import get_single_stocks, analyze_ticker, plot_to_base64

# =========================================================================
# AlphaSpread-style intrinsic value (DCF) per single stock, in tabs.
#
# One tab per stock: market price vs approximate historical intrinsic value.
# Pre-profit / no-FCF names (IONQ, RKLB, quantum, etc.) are shown as
# "DCF N/A" with price only -- DCF is not meaningful without positive FCF.
#
# Tune assumptions here (passed through to the DCF for every stock):
DCF_OVERRIDES = dict(
    # risk_free=0.043, equity_premium=0.05,
    # terminal_growth=0.025, growth_cap=0.25, projection_years=10,
)
# =========================================================================

stocks = get_single_stocks()
print("Analyzing %d holdings (DCF intrinsic value)..." % len(stocks))

results = []
for tk, basket in stocks:
    try:
        v = analyze_ticker(tk, basket, **DCF_OVERRIDES)
    except Exception as e:
        v = {"ticker": tk, "basket": basket, "name": tk, "eligible": False,
             "reason": "error: %s" % e, "price": None, "intrinsic": None,
             "upside_pct": None, "price_dates": [], "price_closes": [],
             "hist_intrinsic": []}
    v["_png"] = plot_to_base64(v)
    results.append(v)
    tag = "OK" if v["eligible"] else "N/A"
    up = ("%+.0f%%" % v["upside_pct"]) if v.get("upside_pct") is not None else "-"
    print("  %-8s %-4s %8s  %s" % (tk, tag, up, v.get("reason", "")))


def _fmt(x, prefix="$", pct=False):
    if x is None:
        return "-"
    if pct:
        return "%+.0f%%" % x
    return "%s%s" % (prefix, format(x, ",.2f"))


def _sort_key(v):
    up = v.get("upside_pct")
    return (0, -up) if (v["eligible"] and up is not None) else (1, 0)


ordered = sorted(results, key=_sort_key)

rows = ""
for v in ordered:
    up = v.get("upside_pct")
    if v["eligible"] and up is not None:
        color = "#137333" if up > 0 else "#c5221f"
        verdict = "Undervalued" if up > 0 else "Overvalued"
        up_cell = '<b style="color:%s">%+.0f%%</b>' % (color, up)
        iv_cell = _fmt(v.get("intrinsic"))
        verdict_cell = '<span style="color:%s">%s</span>' % (color, verdict)
    else:
        up_cell = "-"
        iv_cell = "-"
        verdict_cell = '<span style="color:#999">N/A</span>'
    rows += (
        '<tr><td><b>%s</b></td>'
        '<td style="color:#666">%s</td>'
        '<td>%s</td><td>%s</td>'
        '<td style="text-align:right">%s</td>'
        '<td style="text-align:right;color:#666">%s</td>'
        '<td style="text-align:right;color:#666"><b>%s</b></td>'
        '<td style="text-align:right;color:#666">%s</td>'
        '<td>%s</td></tr>'
    ) % (v["ticker"], v.get("basket", ""), _fmt(v.get("price")),
         iv_cell, up_cell,
         _fmt(v.get("analyst_low")), _fmt(v.get("analyst_mean")),
         _fmt(v.get("analyst_high")), verdict_cell)

summary_html = (
    '<h2 style="font-family:sans-serif">Intrinsic Value - AI Allocation '
    'Portfolio</h2>'
    '<p style="font-family:sans-serif;color:#555;max-width:760px">'
    'DCF (discounted cash flow) intrinsic value vs market price, '
    'AlphaSpread-style. <b>Upside %</b> = how far the market price sits below '
    '(positive) or above (negative) fundamental value. Stocks without positive '
    'free cash flow cannot be valued by DCF and are marked <b>N/A</b>.</p>'
    '<table style="font-family:sans-serif;border-collapse:collapse;'
    'font-size:14px" border="0" cellpadding="7">'
    '<tr style="background:#f1f3f4;text-align:left">'
    '<th>Ticker</th><th>Basket</th><th>Price</th><th>Intrinsic</th>'
    '<th style="text-align:right">Upside</th>'
    '<th style="text-align:right">Analyst low</th>'
    '<th style="text-align:right">Analyst mean</th>'
    '<th style="text-align:right">Analyst high</th>'
    '<th>Verdict</th></tr>'
    + rows +
    '</table>'
    '<p style="font-family:sans-serif;color:#999;font-size:12px">'
    'Historical intrinsic line is approximate (yfinance: ~5 annual statements, '
    'no point-in-time beta/debt). Price line is exact. Not investment advice.</p>'
)

tab_buttons = ""
tab_contents = ""
for i, v in enumerate(results):
    tk = v["ticker"]
    active = " ivActive" if i == 0 else ""
    disp = "block" if i == 0 else "none"
    up = v.get("upside_pct")
    if v["eligible"] and up is not None:
        c = "#137333" if up > 0 else "#c5221f"
        badge = ' <span style="color:%s;font-size:11px">(%+.0f%%)</span>' % (c, up)
    else:
        badge = ' <span style="color:#bbb;font-size:11px">(N/A)</span>'
    tab_buttons += (
        '<button class="ivTab%s" onclick="ivOpen(event,\'iv_%s\')">%s%s</button>'
    ) % (active, tk, tk, badge)

    if v.get("_png"):
        img = ('<img src="data:image/png;base64,%s" '
               'style="max-width:100%%;height:auto">') % v["_png"]
    else:
        img = '<p style="color:#999">No price history available.</p>'

    if v["eligible"]:
        _fcf = v.get("fcf_latest")
        fcf_cell = ("$%sM" % format(_fcf / 1e6, ",.0f")) if _fcf else "n/a"
        dcf_cell = _fmt(v.get("dcf_value")) if v.get("dcf_value") else "n/a (no FCF)"
        _an = v.get("analyst_n")
        an_note = (" (%d analysts)" % _an) if _an else ""
        detail = (
            '<ul style="color:#444;font-size:13px">'
            '<li>Market price: <b>%s</b></li>'
            '<li><b>Intrinsic value (blended): %s</b></li>'
            '<li>&nbsp;&nbsp;Multiples value (fwd EPS x risk-adj P/E): %s</li>'
            '<li>&nbsp;&nbsp;DCF value (normalized FCF): %s</li>'
            '<li>Upside: <b>%s</b></li>'
            '<li>Discount rate (WACC): %.1f%%</li>'
            '<li>Projected growth: %.1f%%/yr (tempered, capped)</li>'
            '<li>Latest FCF: %s</li>'
            '<li style="margin-top:6px">Analyst targets%s: '
            'low <b>%s</b> / mean <b>%s</b> / high <b>%s</b></li></ul>'
            '<p style="color:#999;font-size:11px">Intrinsic = 80%% multiples + '
            '20%% DCF, calibrated to AlphaSpread\'s Base Case. Fair P/E is '
            'risk-adjusted (shrinks as beta rises). Analyst targets are '
            'sell-side price targets from Yahoo Finance.</p>'
        ) % (_fmt(v.get("price")), _fmt(v.get("intrinsic")),
             _fmt(v.get("multiples_value")), dcf_cell,
             _fmt(up, pct=True), v.get("wacc", 0) * 100,
             v.get("growth", 0) * 100, fcf_cell, an_note,
             _fmt(v.get("analyst_low")), _fmt(v.get("analyst_mean")),
             _fmt(v.get("analyst_high")))
    else:
        detail = (
            '<p style="color:#999;font-size:13px"><b>DCF not applicable:</b> '
            '%s. This company has no positive free cash flow, so a discounted '
            'cash-flow valuation would be meaningless. Only the market price '
            'is shown.</p>'
        ) % v.get("reason", "")

    tab_contents += (
        '<div id="iv_%s" class="ivContent" style="display:%s">'
        '<h3 style="font-family:sans-serif">%s - %s '
        '<span style="color:#888;font-weight:normal;font-size:14px">(%s)</span>'
        '</h3>%s%s</div>'
    ) % (tk, disp, tk, v.get("name", ""), v.get("basket", ""), img, detail)

tabs_html = (
    '<style>'
    '.ivTab { background:#f1f3f4; border:none; padding:8px 14px; cursor:pointer;'
    ' font-family:sans-serif; font-size:13px; border-radius:6px; margin:2px; }'
    '.ivTab:hover { background:#e0e0e0; }'
    '.ivTab.ivActive { background:#1a73e8; color:#fff; }'
    '.ivContent { padding:14px 4px; }'
    '</style>'
    '<div style="max-width:1000px">'
    '<h2 style="font-family:sans-serif">Per-Stock Charts</h2>'
    '<div style="display:flex;flex-wrap:wrap;gap:2px;margin-bottom:8px">'
    + tab_buttons +
    '</div>' + tab_contents +
    '</div>'
    '<script>'
    'function ivOpen(evt, id) {'
    ' var c = document.getElementsByClassName("ivContent");'
    ' for (var i=0;i<c.length;i++) c[i].style.display = "none";'
    ' var t = document.getElementsByClassName("ivTab");'
    ' for (var i=0;i<t.length;i++) t[i].className = t[i].className.replace(" ivActive","");'
    ' document.getElementById(id).style.display = "block";'
    ' evt.currentTarget.className += " ivActive";'
    '}'
    '</script>'
)

display(HTML(summary_html))
display(HTML(tabs_html))
